In [1]:
import os
import csv
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import gc
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import joblib

C:\Users\Asus\anaconda3\envs\thesis\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
print(torch.cuda.is_available())

True


In [3]:
# def merge_phase_files(data_path, splits=["train", "valid", "test"], phases=["I", "II", "III"], output_file="all_phases_hint.csv"):
#     merged = []

#     for split in splits:
#         for phase in phases:
#             file = f"phase_{phase}_{split}.csv"
#             path = os.path.join(data_path, file)
#             if os.path.exists(path):
#                 df = pd.read_csv(path)
#                 merged.append(df)
#                 print(f" Loaded {file} with {len(df)} rows")
#             else:
#                 print(f" File not found: {file}")

#     if merged:
#         all_data = pd.concat(merged, ignore_index=True)
#         all_data.to_csv(os.path.join(data_path, output_file), index=False)
#         print(f"\n Merged dataset saved as: {output_file} | Total rows: {len(all_data)}")
#         return os.path.join(data_path, output_file)
#     else:
#         print("No files found to merge.")
#         return None

In [5]:
# merged_file = merge_phase_files(data_path="data/clinical-trial-outcome-prediction/data_reduced")

 Loaded phase_I_train.csv with 5417 rows
 Loaded phase_II_train.csv with 6761 rows
 Loaded phase_III_train.csv with 4165 rows
 Loaded phase_I_valid.csv with 1159 rows
 Loaded phase_II_valid.csv with 1452 rows
 Loaded phase_III_valid.csv with 894 rows
 Loaded phase_I_test.csv with 1160 rows
 Loaded phase_II_test.csv with 1449 rows
 Loaded phase_III_test.csv with 893 rows

 Merged dataset saved as: all_phases_hint.csv | Total rows: 23350


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")
model = AutoModel.from_pretrained("medicalai/ClinicalBERT").to(device)
model.eval()

C:\Users\Asus\anaconda3\envs\thesis\Lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\Asus\anaconda3\envs\thesis\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\Asus\anaconda3\envs\thesis\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\Asus\anaconda3\envs\thesis\Lib\site-packages\transformers\modeling_utils.py:484: FutureWarning: You are using `torch.load` with `weights_only=False` (the current de

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): MultiHeadSelfAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [6]:
def clean_protocol(text):
    if isinstance(text, str):
        lines = text.lower().split('\n')
    elif isinstance(text, list):
        lines = [line.lower() for line in text if isinstance(line, str)]
    else:
        return []

    lines = [line.strip() for line in lines if line.strip()]
    return lines

def split_protocol(protocol):
    lines = clean_protocol(protocol)
    inclusion_idx = exclusion_idx = len(lines)

    for idx, sentence in enumerate(lines):
        if "inclusion criteria" in sentence:
            inclusion_idx = idx
            break

    for idx, sentence in enumerate(lines):
        if "exclusion criteria" in sentence:
            exclusion_idx = idx
            break

    if inclusion_idx < exclusion_idx:
        inclusion = lines[inclusion_idx + 1:exclusion_idx]
        exclusion = lines[exclusion_idx + 1:]
    else:
        inclusion = lines[inclusion_idx + 1:] if inclusion_idx < len(lines) else []
        exclusion = []

    return inclusion, exclusion

def collect_cleaned_sentence_set(input_file):
    df = pd.read_csv(input_file)
    all_sentences = set()
    for protocol in df["Eligibility Criteria"].dropna():
        inclusion, exclusion = split_protocol(protocol)
        all_sentences.update(inclusion + exclusion)
    return all_sentences

In [7]:
class SentenceDataset(Dataset):
    def __init__(self, sentences):
        self.sentences = list(sentences)
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self, idx):
        return self.sentences[idx]

def get_batched_embeddings(sentences, batch_size=64):
    dataset = SentenceDataset(sentences)
    dataloader = DataLoader(dataset, batch_size=batch_size)
    embeddings = []

    for batch in tqdm(dataloader, desc="Embedding in batches"):
        inputs = tokenizer(list(batch), return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        cls_batch = outputs.last_hidden_state[:, 0, :].cpu()
        embeddings.extend(cls_batch)
        del inputs, outputs, cls_batch
        torch.cuda.empty_cache()
        gc.collect()

    return embeddings

In [8]:
def save_sentence_bert_dict_batched(sentences, output_file, batch_size=64, save_every=10000):
    if os.path.exists(output_file):
        sentence2embedding = joblib.load(output_file)
        sentence2embedding = {k: torch.tensor(v) for k, v in sentence2embedding.items()}
        print(f"[Resuming] Loaded existing embeddings: {len(sentence2embedding)}")
    else:
        sentence2embedding = {}

    remaining = [s for s in set(sentences) if s not in sentence2embedding]
    print(f"[Embedding] Remaining sentences: {len(remaining)}")

    for i in range(0, len(remaining), batch_size):
        batch = remaining[i:i + batch_size]
        try:
            batch_embeddings = get_batched_embeddings(batch, batch_size)
            for s, e in zip(batch, batch_embeddings):
                sentence2embedding[s] = e
        except Exception as e:
            print(f"[!] Batch error: {e}")
            continue

        if (i // batch_size) % (save_every // batch_size) == 0 or i + batch_size >= len(remaining):
            numpy_dict = {k: v.numpy() for k, v in sentence2embedding.items()}
            joblib.dump(numpy_dict, output_file)
            print(f"Intermediate save: {len(numpy_dict)} entries")
            del numpy_dict
            gc.collect()

    print(f"Final save: {output_file} | Total embeddings: {len(sentence2embedding)}")
    return sentence2embedding

In [9]:
def protocol2feature(protocol, sentence2vec):
    inclusion, exclusion = split_protocol(protocol)
    inc_vecs = [sentence2vec[s] for s in inclusion if s in sentence2vec]
    exc_vecs = [sentence2vec[s] for s in exclusion if s in sentence2vec]

    # Optional logging
    missing = [s for s in inclusion + exclusion if s not in sentence2vec]
    if missing:
        print(f"Missing sentences: {len(missing)} / {len(inclusion) + len(exclusion)}")

    inc = torch.stack(inc_vecs).mean(0, keepdim=True) if inc_vecs else torch.zeros(1, 768)
    exc = torch.stack(exc_vecs).mean(0, keepdim=True) if exc_vecs else torch.zeros(1, 768)
    return torch.cat([inc, exc], dim=1)

def prepare_criteria_feature(data_path, embedding_path, output_path="criteria"):
    os.makedirs(os.path.join(data_path, output_path), exist_ok=True)
    sentence2vec = joblib.load(os.path.join(data_path, embedding_path))
    sentence2vec = {k: torch.tensor(v) for k, v in sentence2vec.items()}

    for phase in tqdm(["I", "II", "III"], desc="Phases"):
        for split in ["train", "valid", "test"]:
            file = f"phase_{phase}_{split}.csv"
            filepath = os.path.join(data_path, file)
            if not os.path.exists(filepath):
                print(f"File not found: {filepath}")
                continue

            df = pd.read_csv(filepath)
            features = []

            for _, row in df.iterrows():
                protocol = row.get("Eligibility Criteria", "")
                try:
                    feat = protocol2feature(protocol, sentence2vec)
                except Exception as e:
                    print(f"Skipping protocol due to error: {e}")
                    feat = torch.zeros(1, 1536)
                features.append(feat)

            stacked = torch.cat(features, dim=0).numpy()
            out_file = os.path.join(data_path, output_path, f"phase_{phase}_{split}.npy")
            np.save(out_file, stacked)
            print(f"Saved: {out_file} | Shape: {stacked.shape}")

In [ ]:
### def main():
    input_file = "data/clinical-trial-outcome-prediction/data_reduced/all_phases_hint.csv"
    output_pickle = "sentence2embedding.pkl"

    # Step 1: Extract and embed
    sentences = collect_cleaned_sentence_set(input_file)
    print(f"Total unique cleaned sentences: {len(sentences)}")

    save_sentence_bert_dict_batched(
        sentences,
        output_file=os.path.join("data/clinical-trial-outcome-prediction/data", output_pickle),
        batch_size=64,
        save_every=10000
    )

    # Step 2: Save encoded protocol features
    prepare_criteria_feature(data_path="data/clinical-trial-outcome-prediction/data", embedding_path=output_pickle)

if __name__ == "__main__":
    main()

In [12]:
data = np.load("data/clinical-trial-outcome-prediction/data/criteria/phase_I_train.npy")
data

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.08117796,  0.02686832,  0.4507417 , ...,  0.05271117,
         0.10952011, -0.28368452],
       [ 0.08117796,  0.02686832,  0.4507417 , ...,  0.05271117,
         0.10952011, -0.28368452],
       ...,
       [ 0.05291933, -0.04582825,  0.40026248, ...,  0.15450399,
        -0.02775547, -0.2818799 ],
       [ 0.07295403, -0.0500845 ,  0.4759005 , ...,  0.01903476,
         0.0905202 , -0.35055152],
       [ 0.08070333, -0.05328073,  0.528892  , ...,  0.03732948,
         0.10417076, -0.27691352]], dtype=float32)

import csv
from tqdm import tqdm
import numpy as np
from copy import deepcopy
import matplotlib.pyplot as plt
import csv
from functools import reduce
import torch
from torch.autograd import Variable
from rdkit import Chem
import torch.
from torch.utils.data.dataloader import default_collate
import torch.utils.data as data
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from copy import deepcopy